# Catalyst on TPU: smoke test

1. **Runtime → Change runtime type → a TPU** (v5e-1 works). With one TPU chip, the multi-process tests run one process.
2. **Runtime → Run all.** The first run builds a Python 3.12 venv with torch_xla 2.9, which takes a few minutes: Colab's TPU runtime ships no torch_xla, and libtpu for torch_xla 2.9 has no Python 3.13 wheel. Each test runs in its own subprocess, because only one process can hold the TPU.
3. **Read the summary** in the last cell: a PASS/FAIL table, then the environment and failure tracebacks as JSON.

| Test | What it runs | Expected |
|---|---|---|
| `single` | `dl.Engine()` on one XLA device: loss, accuracy, `CheckpointCallback` (`xm.save`), `load_best_on_end` | PASS |
| `ddp` | `dl.DistributedXLAEngine()`: `xmp.spawn(nprocs=None)`, metrics averaged across processes (`xm.mesh_reduce`), collective `xm.save` | PASS |
| `auc`, `auc_ddp` | `single` / `ddp` plus `AUCCallback` | PASS |
| `cifar`, `cifar_ddp` | ResNet9 from catalyst's XLA notebooks, CIFAR10 subset, 1 epoch | PASS |

The synthetic tests also fail if the model does not learn (validation accuracy ≤ 0.5 on 4 easy classes).

In [ ]:
# catalyst to test; switch to neuroneural/catalyst master once the branch is merged
REPO, BRANCH = "paavalipopov/catalyst", "accelerate-xla"
CATALYST = f"git+https://github.com/{REPO}@{BRANCH}"
# (torch, torch_xla, accelerate) stacks; each torch_xla version gets its own venv.
# accelerate < 1.7 needs torch_xla <= 2.6 (xm.xrt_world_size), which has no 3.12 wheels
STACKS = [
    ("2.9.0", "2.9.0", "latest"),
]
TESTS = ["single", "ddp", "auc", "auc_ddp", "cifar", "cifar_ddp"]
TIMEOUT = 1800  # seconds per test

In [ ]:
# a Python 3.12 venv per torch_xla version; the runtime's own packages stay untouched
import json, os, shutil, subprocess, sys

DRY_RUN = os.environ.get("SMOKE_DEVICE") == "cpu"  # CPU dry run, see the script below

PROBE = r"""
import json, os, sys
from importlib.metadata import version
os.environ.setdefault("PJRT_DEVICE", "TPU")
info = {"python": sys.version.split()[0], "torch": version("torch")}
try:
    import torch_xla, torch_xla.runtime as xr
    info.update(torch_xla=torch_xla.__version__, libtpu=version("libtpu"))
    info.update(device_type=xr.device_type(), devices=xr.global_runtime_device_count())
except Exception as ex:
    info["torch_xla_error"] = f"{type(ex).__name__}: {ex}"
try:  # topology from the TPU metadata
    from torch_xla._internal import tpu
    meta = tpu.get_tpu_env()
    info["tpu_env"] = {k: meta.get(k) for k in ("ACCELERATOR_TYPE", "TPU_PROCESS_BOUNDS", "TPU_CHIPS_PER_PROCESS_BOUNDS")}
except Exception as ex:
    info["tpu_env"] = f"{type(ex).__name__}: {ex}"[:200]
print(json.dumps(info))
"""


def sh(*cmd):
    subprocess.run(cmd, check=True)


def make_venv(torch_v, xla_v):
    """Returns the python of a venv with torch, torch_xla[tpu] and catalyst's dependencies."""
    if os.environ.get("SMOKE_PYTHON"):  # dry run: an existing interpreter
        python = os.environ["SMOKE_PYTHON"]
    else:
        venv = os.path.expanduser(f"~/venv-xla-{xla_v}")
        python = f"{venv}/bin/python"
        if not os.path.exists(f"{venv}/ready"):
            shutil.rmtree(venv, ignore_errors=True)
            sh(sys.executable, "-m", "pip", "install", "-q", "-U", "uv")
            sh(sys.executable, "-m", "uv", "venv", "-q", "--seed", "--python", "3.12", venv)
            pip = [python, "-m", "pip", "install", "-q"]
            cpu_index = "https://download.pytorch.org/whl/cpu"
            sh(*pip, f"torch=={torch_v}+cpu", "--extra-index-url", cpu_index)
            # catalyst's dependencies except accelerate; its contrib also needs six
            sh(*pip, f"torch_xla[tpu]=={xla_v}", "numpy", "hydra-slayer", "tqdm", "tensorboardX", "six")
            open(f"{venv}/ready", "w").close()
    # catalyst from the branch on every run, without letting pip touch torch / torch_xla
    sh(python, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", CATALYST)
    return python


PYTHONS, ENVS = {}, {}
for torch_v, xla_v, _ in STACKS:
    if xla_v in PYTHONS:
        continue
    PYTHONS[xla_v] = make_venv(torch_v, xla_v)
    probe = subprocess.run([PYTHONS[xla_v], "-c", PROBE], capture_output=True, text=True)
    try:
        ENVS[xla_v] = json.loads(probe.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        ENVS[xla_v] = {"probe_error": probe.stderr.strip().splitlines()[-3:]}
    print(f"torch_xla {xla_v}:", json.dumps(ENVS[xla_v]))
    with open(f"constraints-{xla_v}.txt", "w") as fout:  # keeps torch for the accelerate installs
        fout.write(f"torch=={ENVS[xla_v].get('torch', torch_v)}\n")
    if ENVS[xla_v].get("device_type") != "TPU" and not DRY_RUN:
        raise RuntimeError(f"torch_xla {xla_v} found no TPU; is the runtime type a TPU?")

In [ ]:
%%writefile catalyst_tpu_smoke.py
"""Catalyst TPU smoke tests. One test per process: python catalyst_tpu_smoke.py <test>.

Tests: single (dl.Engine on one XLA device), ddp (dl.DistributedXLAEngine over all devices),
auc / auc_ddp (single / ddp + AUCCallback), cifar / cifar_ddp (ResNet9 from the catalyst XLA notebooks, CIFAR10 subset).
Prints one line `RESULT {json}`. SMOKE_DEVICE=cpu swaps in CPU engines for a local dry run,
where auc_ddp fails: catalyst's all_gather (AUC under DDP) needs CUDA.
"""
import json
import os
import sys
import tempfile
import traceback
import types

TEST = sys.argv[1]
DEVICE = os.environ.get("SMOKE_DEVICE", "tpu")
if DEVICE == "tpu":
    os.environ.setdefault("PJRT_DEVICE", "TPU")

import torch  # noqa: E402
from torch import nn  # noqa: E402
from torch.utils.data import DataLoader, Subset, TensorDataset  # noqa: E402

from catalyst import dl  # noqa: E402

DISTRIBUTED = TEST.endswith("ddp")


def report(status, **info):
    print("RESULT " + json.dumps({"test": TEST, "status": status, **info}), flush=True)


def make_engine():
    if DEVICE == "cpu":
        if DISTRIBUTED:
            return dl.DistributedDataParallelEngine(
                port=29561, num_node_workers=2, process_group_kwargs={"backend": "gloo"}, cpu=True
            )
        return dl.CPUEngine()
    return dl.DistributedXLAEngine() if DISTRIBUTED else dl.Engine()


def resnet9(in_channels=3, num_classes=10, size=16):
    # same model as catalyst's XLA notebooks
    from catalyst.contrib.layers import ResidualBlock

    def block(cin, cout, pool=False):
        layers = [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
        return nn.Sequential(*layers, *([nn.MaxPool2d(2)] if pool else []))

    s1, s2, s4, s8 = size, size * 2, size * 4, size * 8
    return nn.Sequential(
        block(in_channels, s1),
        block(s1, s2, pool=True),
        ResidualBlock(nn.Sequential(block(s2, s2), block(s2, s2))),
        block(s2, s4, pool=True),
        block(s4, s8, pool=True),
        ResidualBlock(nn.Sequential(block(s8, s8), block(s8, s8))),
        nn.Sequential(nn.MaxPool2d(4), nn.Flatten(), nn.Dropout(0.2), nn.Linear(s8, num_classes)),
    )


def cifar_loaders():
    if DEVICE == "cpu":  # dry run: CIFAR-shaped random data, no download
        x, y = torch.randn(256, 3, 32, 32), torch.randint(0, 10, (256,))
        return DataLoader(TensorDataset(x, y), batch_size=64), DataLoader(TensorDataset(x, y), batch_size=64)
    if not hasattr(torch, "_six"):  # catalyst's CIFAR10 needs it; torch 2.0 removed it
        torch._six = types.SimpleNamespace(string_classes=(str, bytes))
    from catalyst.contrib.data import Compose, ImageToTensor, NormalizeImage
    from catalyst.contrib.datasets import CIFAR10

    tf = Compose([ImageToTensor(), NormalizeImage((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    root = os.path.expanduser("~/cifar")
    train = CIFAR10(root, train=True, download=True, transform=tf)
    valid = CIFAR10(root, train=False, download=True, transform=tf)
    # subsets keep the run to a few minutes
    train, valid = Subset(train, range(4096)), Subset(valid, range(1024))
    return DataLoader(train, batch_size=64, shuffle=True), DataLoader(valid, batch_size=64)


class SmokeRunner(dl.IRunner):
    def __init__(self, logdir):
        super().__init__()
        self._logdir = logdir

    @property
    def num_epochs(self):
        return 1 if TEST.startswith("cifar") else 2

    def get_engine(self):
        return make_engine()

    def get_loggers(self):
        return {"csv": dl.CSVLogger(logdir=self._logdir)}

    def get_loaders(self):
        if TEST.startswith("cifar"):
            train, valid = cifar_loaders()
            return {"train": train, "valid": valid}
        g = torch.Generator().manual_seed(0)
        x = torch.randn(512, 16, generator=g)
        y = (x[:, 0] > 0).long() + 2 * (x[:, 1] > 0).long()
        data = TensorDataset(x, y)
        return {
            "train": DataLoader(data, batch_size=32, shuffle=True),
            "valid": DataLoader(data, batch_size=64),
        }

    def get_model(self):
        torch.manual_seed(0)
        if TEST.startswith("cifar"):
            return resnet9()
        return nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 4))

    def get_criterion(self):
        return nn.CrossEntropyLoss()

    def get_optimizer(self, model):
        return torch.optim.Adam(model.parameters(), lr=1e-2)

    def get_scheduler(self, optimizer):
        return None

    def get_callbacks(self):
        callbacks = {
            "criterion": dl.CriterionCallback(
                input_key="logits", target_key="targets", metric_key="loss"
            ),
            "backward": dl.BackwardCallback(metric_key="loss"),
            "optimizer": dl.OptimizerCallback(metric_key="loss"),
            "accuracy": dl.AccuracyCallback(input_key="logits", target_key="targets", topk=(1,)),
            "checkpoint": dl.CheckpointCallback(
                self._logdir,
                loader_key="valid",
                metric_key="loss",
                minimize=True,
                topk=1,
                load_best_on_end=True,
            ),
        }
        if TEST.startswith("auc"):
            callbacks["auc"] = dl.AUCCallback(input_key="logits", target_key="targets")
        return callbacks

    def handle_batch(self, batch):
        x, y = batch
        self.batch = {"features": x, "targets": y, "logits": self.model(x)}

    def on_experiment_end(self, runner):
        dtype = self.engine.distributed_type
        info = {
            "rank": int(self.engine.process_index),
            "world_size": int(self.engine.num_processes),
            "distributed_type": getattr(dtype, "value", str(dtype)),
            "device": str(self.engine.device),
            "valid": {k: float(v) for k, v in self.epoch_metrics["valid"].items()},
        }
        with open(os.path.join(self._logdir, f"rank{info['rank']}.json"), "w") as fout:
            json.dump(info, fout)
        super().on_experiment_end(runner)


def main():
    logdir = tempfile.mkdtemp(prefix=f"smoke_{TEST}_")
    try:
        SmokeRunner(logdir).run()

        # every rank reports; DDP-averaged metrics must agree across ranks
        files = sorted(f for f in os.listdir(logdir) if f.startswith("rank"))
        infos = [json.load(open(os.path.join(logdir, f))) for f in files]
        assert infos, "no rank wrote a report"
        world_size = infos[0]["world_size"]
        assert len(infos) == world_size, f"{len(infos)} rank reports for world_size {world_size}"
        losses = {round(i["valid"]["loss"], 5) for i in infos}
        assert len(losses) == 1, f"ranks disagree on the valid loss: {sorted(losses)}"
        assert os.path.isfile(os.path.join(logdir, "model.best.pth")), "no model.best.pth"
        # the 4 synthetic classes are easy; a model that does not learn stays near 0.25
        accuracy = infos[0]["valid"]["accuracy01"]
        assert TEST.startswith("cifar") or accuracy > 0.5, f"no learning: accuracy01={accuracy:.3f}"

        report(
            "PASS",
            world_size=world_size,
            distributed_type=infos[0]["distributed_type"],
            device=infos[0]["device"],
            valid={k: round(v, 4) for k, v in infos[0]["valid"].items()},
        )
    except BaseException as ex:  # noqa: B902 - report every failure, including SystemExit from workers
        lines = traceback.format_exc().strip().splitlines()
        report("FAIL", error=f"{type(ex).__name__}: {ex}"[:600], traceback_tail=lines[-6:])
        sys.exit(1)


if __name__ == "__main__":
    main()


In [ ]:
# every test on every stack
# Colab's v5e-1 reports 2,2,1 chips per process (a 4-chip host), which breaks
# xmp.spawn; multi-process tests get torch_xla's own one-chip topology instead
ONE_CHIP = {"TPU_PROCESS_BOUNDS": "1,1,1", "TPU_CHIPS_PER_PROCESS_BOUNDS": "1,1,1"}
results = []
for torch_v, xla_v, acc in STACKS:
    python = PYTHONS[xla_v]
    spec = "accelerate" if acc == "latest" else f"accelerate=={acc}"
    sh(python, "-m", "pip", "install", "-q", "--upgrade", "-c", f"constraints-{xla_v}.txt", spec)
    version = subprocess.run([python, "-c", "import accelerate; print(accelerate.__version__)"],
                             capture_output=True, text=True).stdout.strip()

    for test in TESTS:
        print(f"torch_xla {xla_v} | accelerate {version} | {test} ...", flush=True)
        one_chip = ENVS[xla_v].get("devices") == 1 and test.endswith("ddp")
        try:
            proc = subprocess.run([python, "catalyst_tpu_smoke.py", test],
                                  env={**os.environ, **ONE_CHIP} if one_chip else None,
                                  capture_output=True, text=True, timeout=TIMEOUT)
            lines = [l for l in proc.stdout.splitlines() if l.startswith("RESULT ")]
            if lines:
                res = json.loads(lines[-1][len("RESULT "):])
            else:
                res = {"test": test, "status": "FAIL", "error": "no RESULT line",
                       "stderr_tail": proc.stderr.strip().splitlines()[-8:]}
        except subprocess.TimeoutExpired:
            res = {"test": test, "status": "FAIL", "error": f"timeout after {TIMEOUT}s"}

        res.update(torch_xla=xla_v, accelerate=version)
        results.append(res)
        print("   ->", res["status"], "" if res["status"] == "PASS" else res.get("error", "")[:300])

In [ ]:
# summary: PASS/FAIL table, then the environment and failure details as JSON
print(f"{'torch_xla':<10} {'accelerate':<11} {'test':<10} {'status':<6} world  info")
for r in results:
    if r["status"] == "FAIL":
        info = r.get("error", "")
    else:
        info = f"{r.get('distributed_type', '')}, valid accuracy {r['valid']['accuracy01']:.3f}"
    print(f"{r['torch_xla']:<10} {r['accelerate']:<11} {r['test']:<10} {r['status']:<6} "
          f"{str(r.get('world_size', '-')):<6} {info[:120]}")
print()
print(json.dumps({"env": ENVS, "results": results}))